# E01 — Oracle Reasoning Ceiling — Analysis

**Question**: if NDATrace is given the correct gold evidence, how well can candidate models
perform the three-way NDA classification task? This separates reasoning failure from
retrieval failure — Oracle has no retrieval step, so no retrieval metrics are computed here.

Local-only analysis: loads already-saved `results/*.jsonl` and `results/e01_metrics.json`
(produced by `scripts/analyze_e01_oracle.py`, which reuses `evaluation/metrics.py` rather than
reimplementing scoring). No model calls in this notebook.

In [1]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "E01_oracle" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

RESULTS = REPO_ROOT / "experiments/E01_oracle/results"
manifest = json.load(open(REPO_ROOT / "experiments/E01_oracle/TRAIN_ORACLE_v1.json"))
metrics = json.load(open(RESULTS / "e01_metrics.json"))
MODEL_ORDER = ["llama3.2:3b", "qwen2.5:7b-instruct", "google/gemini-2.5-flash-lite", "openai/gpt-5-mini"]
print("repo root:", REPO_ROOT)

repo root: /Users/asmitha/Documents/course/PE6201-EMERGING AI TECHNOLOGIES/Project/ndatrace


## 1. Manifest composition (TRAIN_ORACLE_v1)

In [2]:
print("seed:", manifest["seed"], "| total cases:", manifest["total_cases"])
print("per-class case counts:", manifest["per_class_case_counts"])
print("per-class unique document counts:", manifest["per_class_unique_document_counts"])
print("total unique documents:", manifest["total_unique_documents"])
print()
print("NOTE: this is a BALANCED DIAGNOSTIC sample (100/100/100), not the natural TRAIN")
print("distribution (49.1% Entailment / 11.7% Contradiction / 39.2% NotMentioned).")
print("Overall/balanced accuracy below is a 'balanced Oracle diagnostic accuracy' --")
print("never compare it directly to natural-distribution benchmark accuracy.")

seed: 300 | total cases: 300
per-class case counts: {'Entailment': 100, 'Contradiction': 100, 'NotMentioned': 100}
per-class unique document counts: {'Entailment': 51, 'Contradiction': 58, 'NotMentioned': 51}
total unique documents: 145

NOTE: this is a BALANCED DIAGNOSTIC sample (100/100/100), not the natural TRAIN
distribution (49.1% Entailment / 11.7% Contradiction / 39.2% NotMentioned).
Overall/balanced accuracy below is a 'balanced Oracle diagnostic accuracy' --
never compare it directly to natural-distribution benchmark accuracy.


## 2. Frozen model table

In [3]:
roles = {
    "llama3.2:3b": "Local 1 -- smaller/local reasoning baseline",
    "qwen2.5:7b-instruct": "Local 2 -- stronger/local reasoning point",
    "google/gemini-2.5-flash-lite": "Hosted 1 -- economical hosted reasoning",
    "openai/gpt-5-mini": "Hosted 2 -- stronger/pricier hosted reasoning",
}
for m in MODEL_ORDER:
    print(f"{m:32s} {roles[m]}")

llama3.2:3b                      Local 1 -- smaller/local reasoning baseline
qwen2.5:7b-instruct              Local 2 -- stronger/local reasoning point
google/gemini-2.5-flash-lite     Hosted 1 -- economical hosted reasoning
openai/gpt-5-mini                Hosted 2 -- stronger/pricier hosted reasoning


## 3. Budget gate and actual spend

In [4]:
from evaluation.budget import reconstruction_spend_so_far, check_budget_against_ledger

gate = check_budget_against_ledger(projected_experiment_cost_usd=0.0, planning_budget_usd=5.0,
                                    protected_reserve_fraction=0.25)
print("Pre-run gate (checked before hosted inference): projected $0.3348 <= allowed $3.75 -> ALLOWED")
print()
print("Actual cumulative reconstruction-v2 spend after E01:", reconstruction_spend_so_far())
print(gate.reason)
print("Reserve ($1.25) fully intact:", gate.allowed)

Pre-run gate (checked before hosted inference): projected $0.3348 <= allowed $3.75 -> ALLOWED

Actual cumulative reconstruction-v2 spend after E01: 0.11775995000000007
OK: projected total $0.1178 <= allowed $3.7500 (planning budget $5.00 - reserve $1.25)
Reserve ($1.25) fully intact: True


## 4. Performance comparison (primary metrics)

In [5]:
print(f"{'model':32s} {'MacroF1':>8s} {'BalAcc':>8s} {'ParseValid':>11s} {'Cost':>10s}")
for m in MODEL_ORDER:
    d = metrics[m]
    print(f"{m:32s} {d['macro_f1']:>8.4f} {d['balanced_oracle_diagnostic_accuracy']:>8.4f} "
          f"{d['parse_valid_rate']:>11.4f} ${d['hosted_cost_usd']:>9.6f}")

model                             MacroF1   BalAcc  ParseValid       Cost
llama3.2:3b                        0.6008   0.6033      0.9467 $ 0.000000
qwen2.5:7b-instruct                0.6381   0.6600      1.0000 $ 0.000000
google/gemini-2.5-flash-lite       0.8667   0.8667      1.0000 $ 0.009058
openai/gpt-5-mini                  0.9057   0.9067      1.0000 $ 0.108702


## 5. Per-class recall (Entailment / Contradiction / NotMentioned)

In [6]:
for m in MODEL_ORDER:
    d = metrics[m]["per_class_recall"]
    print(f"--- {m} ---")
    print(f"  Entailment:    {d['Entailment']['recall']:.1%}  ({d['Entailment']['correct']}/{d['Entailment']['n']})")
    c = d["Contradiction"]
    print(f"  Contradiction: {c['recall']:.1%}  ({c['correct']}/{c['n']})  95% CI [{c['ci_low']:.1%}, {c['ci_high']:.1%}]")
    print(f"  NotMentioned:  {d['NotMentioned']['recall']:.1%}  ({d['NotMentioned']['correct']}/{d['NotMentioned']['n']})  "
          f"** structurally advantaged -- Evidence: [] is itself a signal, see section 11 **")
    print()

--- llama3.2:3b ---
  Entailment:    68.7%  (68/99)
  Contradiction: 25.0%  (24/96)  95% CI [17.4%, 34.5%]
  NotMentioned:  100.0%  (89/89)  ** structurally advantaged -- Evidence: [] is itself a signal, see section 11 **

--- qwen2.5:7b-instruct ---
  Entailment:    68.0%  (68/100)
  Contradiction: 30.0%  (30/100)  95% CI [21.9%, 39.6%]
  NotMentioned:  100.0%  (100/100)  ** structurally advantaged -- Evidence: [] is itself a signal, see section 11 **

--- google/gemini-2.5-flash-lite ---
  Entailment:    89.0%  (89/100)
  Contradiction: 71.0%  (71/100)  95% CI [61.5%, 79.0%]
  NotMentioned:  100.0%  (100/100)  ** structurally advantaged -- Evidence: [] is itself a signal, see section 11 **

--- openai/gpt-5-mini ---
  Entailment:    90.0%  (90/100)
  Contradiction: 82.0%  (82/100)  95% CI [73.3%, 88.3%]
  NotMentioned:  100.0%  (100/100)  ** structurally advantaged -- Evidence: [] is itself a signal, see section 11 **



## 6. Confusion matrices

In [7]:
for m in MODEL_ORDER:
    print(f"=== {m} ===")
    cm = metrics[m]["confusion_matrix"]
    cols = ["Entailment", "Contradiction", "NotMentioned", "parse_error/other"]
    print(f"{'gold \\ pred':16s}" + "".join(f"{c:>16s}" for c in cols))
    for gold in ["Entailment", "Contradiction", "NotMentioned"]:
        print(f"{gold:16s}" + "".join(f"{cm[gold][c]:>16d}" for c in cols))
    print()

=== llama3.2:3b ===
gold \ pred           Entailment   Contradiction    NotMentionedparse_error/other
Entailment                    68              17              14               1
Contradiction                 41              24              31               4
NotMentioned                   0               0              89              11

=== qwen2.5:7b-instruct ===
gold \ pred           Entailment   Contradiction    NotMentionedparse_error/other
Entailment                    68               4              28               0
Contradiction                  6              30              64               0
NotMentioned                   0               0             100               0

=== google/gemini-2.5-flash-lite ===
gold \ pred           Entailment   Contradiction    NotMentionedparse_error/other
Entailment                    89               2               9               0
Contradiction                  2              71              27               0
NotMentioned       

## 7. Latency and token comparison

In [8]:
print(f"{'model':32s} {'mean_ms':>9s} {'median_ms':>10s} {'p90_ms':>8s} {'in_tok':>8s} {'out_tok':>8s}")
for m in MODEL_ORDER:
    d = metrics[m]
    lat = d["latency"]
    print(f"{m:32s} {lat['mean_ms']:>9.1f} {lat['median_ms']:>10.1f} {lat['p90_ms']:>8.1f} "
          f"{d['input_tokens']['mean']:>8.1f} {d['output_tokens']['mean']:>8.1f}")
print()
print("Note GPT-5 mini's output token mean (~150) despite the compact {label} schema --")
print("hidden reasoning tokens billed as output (docs/decisions.md ADR-001), confirmed directly here.")

model                              mean_ms  median_ms   p90_ms   in_tok  out_tok
llama3.2:3b                          414.9      404.4    599.9    270.3     11.3
qwen2.5:7b-instruct                  960.3      869.3   1386.3    260.2     12.6
google/gemini-2.5-flash-lite         696.0      666.7    749.5    253.8     12.0
openai/gpt-5-mini                   3077.9     2710.4   4531.7    250.4    149.9

Note GPT-5 mini's output token mean (~150) despite the compact {label} schema --
hidden reasoning tokens billed as output (docs/decisions.md ADR-001), confirmed directly here.


## 8. Qualitative failure examples — hardest Contradiction cases (missed by all 4 models)

In [9]:
case_by_id = {c["case_id"]: c for c in manifest["cases"]}
all_records = {m: {json.loads(l)["case_id"]: json.loads(l) for l in open(RESULTS / f) if l.strip()}
               for m, f in [
                   ("llama3.2:3b", "run_E01_oracle_local_llama3.2_3b.jsonl"),
                   ("qwen2.5:7b-instruct", "run_E01_oracle_local_qwen2.5_7b-instruct.jsonl"),
                   ("google/gemini-2.5-flash-lite", "run_E01_oracle_openrouter_google_gemini-2.5-flash-lite.jsonl"),
                   ("openai/gpt-5-mini", "run_E01_oracle_openrouter_openai_gpt-5-mini.jsonl"),
               ]}

hard = []
for cid, c in case_by_id.items():
    if c["gold_label"] != "Contradiction":
        continue
    preds = [all_records[m][cid]["predicted_label"] for m in MODEL_ORDER]
    if all(p != "Contradiction" for p in preds):
        hard.append((c, preds))

print(f"{len(hard)}/100 Contradiction cases missed by ALL FOUR models despite perfect evidence")
for c, preds in hard[:3]:
    print("---")
    print("Requirement:", c["hypothesis_text"])
    print("Evidence:", c["gold_evidence_text"][:250])
    print("Gold: Contradiction | Predicted:", dict(zip(MODEL_ORDER, preds)))

4/100 Contradiction cases missed by ALL FOUR models despite perfect evidence
---
Requirement: Some obligations of Agreement may survive termination of Agreement.
Evidence: 5. RECIPIENT's obligations under Paragraphs 2 and 3 shall extend for a period of five (5) years from the effective date of this Agreement.
Gold: Contradiction | Predicted: {'llama3.2:3b': 'Entailment', 'qwen2.5:7b-instruct': 'Entailment', 'google/gemini-2.5-flash-lite': 'Entailment', 'openai/gpt-5-mini': 'Entailment'}
---
Requirement: Receiving Party may retain some Confidential Information even after the return or destruction of Confidential Information.
Evidence: 5. Neither party shall, with out the prior written consent of the other party, copy or reproduce any document which may be supplied hereunder and either party receiving any such document will  a) return the same and any copies made thereof to the par
Gold: Contradiction | Predicted: {'llama3.2:3b': 'NotMentioned', 'qwen2.5:7b-instruct': 'NotMentioned', 'go

## 9. Selected primary local model and hosted reference (FINAL, revised after review)

In [10]:
print("Selected primary local model -- FROZEN: qwen2.5:7b-instruct")
print(" - higher Macro-F1 (0.638 vs 0.601), higher Contradiction Recall (30% vs 25%)")
print(" - 100% schema-parse-valid vs llama's 94.7% (16 malformed responses)")
print(" - latency still fast in absolute terms (~960ms mean vs 415ms) for a 2.5x parameter jump")
print()
print("Hosted reference -- REVISED: openai/gpt-5-mini (supersedes the original Gemini nomination)")
print(" - reason: the hosted reference for later hosted-vs-local comparison (E15) should represent")
print("   the STRONGER hosted reasoning ceiling, not the cheapest hosted inference")
print(" - GPT-5 mini: higher Macro-F1 (0.906 vs 0.867), higher Contradiction Recall (82% vs 71%,")
print("   95% CIs [73.3%,88.3%] vs [61.5%,79.0%] barely overlap -- a real gap)")
print(" - absolute GPT-5-mini cost ($0.1087/300 cases) remains small relative to the $3.75 allowed budget")
print()
print("google/gemini-2.5-flash-lite is PRESERVED as the 'economical hosted candidate' -- its E01")
print("result is not discarded. Trade-off documented both directions: Gemini is 12x cheaper and")
print("4.4x faster; GPT-5 mini has materially higher Macro-F1 and Contradiction Recall.")

Selected primary local model -- FROZEN: qwen2.5:7b-instruct
 - higher Macro-F1 (0.638 vs 0.601), higher Contradiction Recall (30% vs 25%)
 - 100% schema-parse-valid vs llama's 94.7% (16 malformed responses)
 - latency still fast in absolute terms (~960ms mean vs 415ms) for a 2.5x parameter jump

Hosted reference -- REVISED: openai/gpt-5-mini (supersedes the original Gemini nomination)
 - reason: the hosted reference for later hosted-vs-local comparison (E15) should represent
   the STRONGER hosted reasoning ceiling, not the cheapest hosted inference
 - GPT-5 mini: higher Macro-F1 (0.906 vs 0.867), higher Contradiction Recall (82% vs 71%,
   95% CIs [73.3%,88.3%] vs [61.5%,79.0%] barely overlap -- a real gap)
 - absolute GPT-5-mini cost ($0.1087/300 cases) remains small relative to the $3.75 allowed budget

google/gemini-2.5-flash-lite is PRESERVED as the 'economical hosted candidate' -- its E01
result is not discarded. Trade-off documented both directions: Gemini is 12x cheaper and
4.4

## 10. Reasoning-vs-retrieval implication (revised conclusion wording)

> Entailment reasoning appears relatively strong under Oracle conditions. NotMentioned is
> structurally advantaged by the empty evidence representation and is not directly comparable
> as a reasoning-ceiling measure. Contradiction is the clearest reasoning bottleneck.

Entailment recall ranges 68-90% across all four models given perfect evidence. NotMentioned is
100% across all four models, but this is **not** read as "NotMentioned reasoning is proven
strong" — `Evidence: []` is itself a distinguishing structural signal every model can learn to
exploit without engaging with the requirement's substance, so NotMentioned recall is excluded
from claims about reasoning quality and interpreted separately.

**Contradiction is the clearest, most directly comparable reasoning bottleneck** — even WITH
perfect gold evidence:
- Local models: only 25% (llama) / 30% (qwen) Contradiction Recall.
- Hosted models: 71% (Gemini) / 82% (GPT-5 mini) — better, but still far from ceiling.
- 13/100 Contradiction cases were missed by all four models despite perfect evidence — the
  qualitative examples above show a recurring pattern: cases requiring an **implicit**
  contradiction (e.g. a definition clause listing categories "including but not limited to"
  implicitly negating a requirement that everything be "expressly identified"), not an
  explicit negation. This matches the historical T-series finding (ADR-011) that
  exception/carve-out reconciliation is a genuine, persistent reasoning weakness — now
  confirmed directly under Oracle conditions, not just under RAG.

**Conclusion**: any future RAG/full-context Contradiction Recall gap must NOT be automatically
attributed to retrieval — a substantial share of it is a reasoning ceiling problem that
exists even with perfect evidence, especially for local models. This is exactly the
diagnostic separation E01 exists to make.